In [ ]:
import re, collections, requests, html, pycallnumber as pycn
from lxml import etree

marc_rda_relators = {
'''Creates dictionary of relator codes common to MARC and RDA'''
    "ape": "appellee",
    "apl": "appellant",
    "arc": "architect",
    "art": "artist",
    "aup": "audio producer",
    "aus": "screenwriter",
    "aut": "author",
    "bka": "book artist",
    "cad": "casting director",
    "chr": "choreographer",
    "cll": "calligrapher",
    "cmp": "composer",
    "com": "compiler",
    "cou": "court governed",
    "csl": "consultant",
    "ctg": "cartographer",
    "dfd": "defendant",
    "dgc": "degree committee member",
    "dgg": "degree granting institution",
    "dgs": "degree supervisor",
    "drt": "director",
    "dsr": "designer",
    "dte": "dedicatee",
    "dto": "dedicator",
    "edd": "editorial director",
    "enj": "enacting jurisdiction",
    "fmd": "film director",
    "fmk": "filmmaker",
    "fmp": "film producer",
    "his": "host institution",
    "inv": "inventor",
    "isb": "issuing body",
    "ive": "interviewee",
    "ivr": "interviewer",
    "jud": "judge",
    "jug": "jurisdiction governed",
    "lbt": "librettist",
    "lsa": "landscape architect",
    "lyr": "lyricist",
    "med": "medium",
    "orm": "organizer",
    "pht": "photographer",
    "pra": "praeses",
    "prg": "programmer",
    "prn": "production company",
    "pro": "producer",
    "ptf": "plaintiff",
    "rap": "rapporteur",
    "rcp": "addressee",
    "rdd": "radio director",
    "res": "researcher",
    "rpc": "radio producer",
    "rsp": "respondent",
    "rxa": "remix artist",
    "scl": "sculptor",
    "tld": "television director",
    "tlp": "television producer",
    }

'''Gets xml file and sets tree and root'''
tree = etree.parse(r"C:\Users\yello\OneDrive\Documents\EAD2MARC\EAD2MARC_workzone\MC122_ead3.xml")
root = tree.getroot()

'''Checks that document is EAD3'''
if 'http://ead3.archivists.org/schema/' in root.tag:

    '''Get target <c> tag'''
    # (This portion of code was partially revised utilizing ChatGPT 5.2)
    target_id = input("Enter aspace id of target <c>: ").strip()

    result = root.xpath(
        "//*[starts-with(local-name(), 'c') and @id=$t_id]",
        t_id=target_id
    )
    # TODO: change to <c> tags specifically instead of any tag that starts with c??

    '''Set global variables'''
    c0_raw = result[0]
    c0_print = etree.tostring(c0_raw, pretty_print=True, encoding="unicode")

    vaid_fetch = root.xpath(".//*[local-name()='recordid']")
    if vaid_fetch:
        vaid_raw = vaid_fetch[0]
        vaid_fetch = vaid_raw.xpath(".//*[local-name()='descriptivenote']")
        vaid_clean = vaid_raw.xpath("string()").strip(".")
        vaid_head_list = vaid_raw.xpath(".//*[local-name()='head']")
        if vaid_head_list:
            vaid_head = vaid_head_list[0].xpath("string()").strip(".")
            vaid_clean = vaid_clean.replace(vaid_head, "")
        vaid_clean = html.escape(vaid_clean)

    '''Set global names lists'''
    names_list = c0_raw.xpath(".//*[local-name()='origination']") # (Troubleshot using ChatGPT 5.2)

    all_persnames_list = []
    all_corpnames_list = []
    all_famnames_list = []

    creator_names_list = []
    creator_persnames_list = []
    creator_corpnames_list = []
    creator_famnames_list = []

    subject_names_list = []
    subject_persnames_list = []
    subject_corpnames_list = []
    subject_famnames_list = []

    source_names_list = []
    source_persnames_list = []
    source_corpnames_list = []
    source_famnames_list = []

    for orig in names_list:
        if orig.attrib["label"].lower() == "creator":
            creator_names_list.append(orig)
        elif orig.attrib["label"].lower() == "subject":
            subject_names_list.append(orig)
        elif orig.attrib["label"].lower() == "source":
            source_names_list.append(orig)

    for orig in names_list:
        if orig.xpath(".//*[local-name()='persname']"):
            all_persnames_list.append(orig)
            if orig.attrib["label"].lower() == "creator":
                creator_persnames_list.append(orig)
            elif orig.attrib["label"].lower() == "subject":
                subject_persnames_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_persnames_list.append(orig)
        elif orig.xpath(".//*[local-name()='corpname']"):
            all_corpnames_list.append(orig)
            if orig.attrib["label"].lower() == "creator":
                creator_corpnames_list.append(orig)
            elif orig.attrib["label"].lower() == "subject":
                subject_corpnames_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_corpnames_list.append(orig)
        elif orig.xpath(".//*[local-name()='famname']"):
            all_famnames_list.append(orig)
            if orig.attrib["label"].lower() == "creator":
                creator_famnames_list.append(orig)
            elif orig.attrib["label"].lower() == "subject":
                subject_famnames_list.append(orig)
            elif orig.attrib["label"].lower() == "source":
                source_famnames_list.append(orig)

    # TODO: ADD SUPPORT FOR OTHER ASPACE AGENT TYPES (FAMILY, SOFTWARE)

    print (c0_print)
else:
    '''Returns error message if document is not EAD3'''
    print("Uploaded file must be in EAD3 (not EAD 2002 or any other EAD version). Please upload a new file and try again.")


In [ ]:
def callnotest():
    unitid_list = c0_raw.xpath(".//*[local-name()='unitid']")
    testlist = []
    for unitid in unitid_list:
        unitid_str = unitid.xpath("string()").strip()
        # (Troubleshot with Claud Opus 4.6)
        callno = pycn.callnumber(unitid_str)
        callno_type = type(callno).__name__
        testlist.append(callnotest)
        if callno_type == "LC":
            print(f"LC: {unitid_str}")
        elif callno_type == "Dewey":
            print(f"Dewey: {unitid_str}")
        else:
            print(f"{callno_type}: {unitid_str}")
    return testlist

callnotest()